In [2]:
import sys
print(sys.executable)

C:\Users\1\anaconda3\python.exe


In [3]:
import requests
import pandas as pd
import numpy as np
import matplotlib
import os

In [4]:
	
# Toronto Open Data is stored in a CKAN instance. It's APIs are documented here:	
# https://docs.ckan.org/en/latest/api/
# To hit our Toronto Data API, you'll be making requests to:
	
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

In [5]:
# Datasets are called "packages". Each package can contain many "resources"
	
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
	
package_url = base_url + "/api/3/action/package_show"
params = {"wardprofile":{ "id": "ward-profiles-25-ward-model"}, "neighbourhoods":{ "id": "neighbourhoods"}, "watermains": { "id": "watermains"}, "watermainbreaks" :{ "id": "watermain-breaks"}, "311":{ "id": "311-service-requests-customer-initiated"}, "climate": { "id": "current-and-future-climate"}}



In [9]:
# To get resource data + download files
def get_files_from_package(base_url, param):
    package = requests.get(package_url, params=param).json()
    files = []
    if not package.get("success"):
        print("API request failed:")
        print(package.get("error"))
    else:
        for idx, resource in enumerate(package["result"]["resources"]):
            # To get metadata for non datastore_active resources:
            if not resource["datastore_active"]:

                resource_url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
                resource_metadata = requests.get(resource_url).json()

                #print(resource_metadata)

                result = resource_metadata["result"]
                file_format = result.get("format", "").lower()

                if file_format in ["csv", "xls", "xlsx"]:
                    file_url = result["url"]

                    # ✅ extract filename
                    file_name = file_url.split("/")[-1].split("?")[0]

                    # ✅ define save path (your existing raw folder)
                    file_path = os.path.join("data", "raw", file_name)

                    print(f"Downloading {file_name} → {file_path}")

                    try:
                        response = requests.get(file_url)
                        response.raise_for_status()

                        # ✅ save file
                        with open(file_path, "wb") as f:
                            f.write(response.content)

                        files.append({
                            "name": file_name,
                            "url": file_url,
                            "path": file_path
                        })

                    except Exception as e:
                        print(f"Failed to download {file_name}: {e}")

    return files

In [32]:
#dfs.keys()

In [10]:
all_files = {}
for key, param in params.items():
    #print(key)
    #print(requests.get(package_url, params = param).json())
    files = get_files_from_package(base_url, param)
    #print(files)
    all_files[key] = files

In [94]:
#Parse the data from the api links and check if they are compatible datatypes, then store them in a list to be further processed. Cleam
def convertData(files):
    data = []
    for f in files:
        filename = f['name']
        file_url = f['url']
        try:
            if f['url'].lower().endswith(".csv"):
                df = pd.read_csv(file_url)
            elif f['url'].lower().endswith((".xls", ".xlsx")):
                df = pd.read_excel(file_url, engine="openpyxl")
            else:
                print(f"Skipping unsupported file type: {file_url}")
                continue
                
            data.append({
                "name": f['name'],
                "df": df})
        except Exception as e:
            print(f"Failed to load {file_name}: {e}")
    
    #create a dictionary of the dataframes in the data, data will be accessed via the keys of dfs **dfs.keys()**
    dfs = {item["name"]: item["df"] for item in data}
    return dfs